# Asistente RAG para la Mesa de Ayuda de TI — Andina Tech (ficticia)
**Sistemas Inteligentes Computacionales · Parcial práctico RAG (Corte 2)**

El asistente responde preguntas sobre manuales, políticas, preguntas frecuentes y el catálogo de servicios de una Mesa de Ayuda, **cita** la evidencia con archivo y localizador, y **se abstiene** cuando la evidencia es insuficiente.

- **Usuarios:** analistas de nivel 1 de la Mesa de Ayuda y empleados en autoservicio.
- **Preguntas que resuelve:** VPN (errores y uso), cuentas y contraseñas, MFA, correo, impresión, prioridades y SLA, solicitudes de servicio, seguridad del puesto.
- **Fuera de alcance:** costos y licencias, servidores y bases de datos, redes del hogar, desarrollo de software, acciones sobre sistemas.
- **Datos:** corpus **sintético** (identificado como tal en cada archivo).

Orden del cuaderno: 1) instalación y credenciales → 2) corpus e índices → 3) demo de consulta → 4) preguntas e hipótesis → 5) parte A (36 corridas) → 6) valoración y métricas → 7) parte B.

## 1. Instalación

In [ ]:
# Si está en Colab: suba rag_mesa_ayuda.zip al panel de archivos (o clone su repositorio) y ejecute esta celda.
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("rag_mesa_ayuda").exists() and pathlib.Path("rag_mesa_ayuda.zip").exists():
    !unzip -q rag_mesa_ayuda.zip
if pathlib.Path("rag_mesa_ayuda").exists():
    os.chdir("rag_mesa_ayuda")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

## 2. Credenciales
La clave **nunca** se escribe en el cuaderno. Se toma, en este orden, de: los *Secretos* de Colab (`OPENROUTER_API_KEY`), el archivo `.env` local (ver `.env.example`) o una entrada oculta con `getpass`. No se imprime.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv(".env")

if not os.getenv("OPENROUTER_API_KEY") and os.getenv("RAG_TEST_MODE") != "1":
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")
print("Clave OpenRouter configurada:", bool(os.getenv("OPENROUTER_API_KEY")))

### Verificar modelos disponibles (antes de las corridas finales)
Los modelos gratuitos de OpenRouter rotan. Esta celda confirma que los identificadores configurados existen. Si alguno no está, cámbielo con `os.environ["LLM_MODEL"] = "..."` **antes** de importar `rag` (o en `.env`) y no lo cambie después.

In [ ]:
import requests
if os.getenv("RAG_TEST_MODE") != "1":
    modelos = {m["id"] for m in requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]}
    for var, default in [("LLM_MODEL", "meta-llama/llama-3.3-70b-instruct:free"), ("LLM_MODEL_ALT", "google/gemma-3-27b-it:free")]:
        mid = os.getenv(var, default)
        print(f"{var:14s} {mid:45s} {'✔ disponible' if mid in modelos else '✘ NO disponible'}")
    print("\nAlgunos modelos gratuitos actuales:", sorted(m for m in modelos if m.endswith(":free"))[:25])

## 3. Configuraciones y modelos registrados

In [ ]:
import pandas as pd
from rag.config import CONFIGS
from rag.pipeline import RAGSystem, format_result

cols = ["descripcion","use_rag","chunking","chunk_size","chunk_overlap","retrieval","k","embedding_model",
        "chunk_tokenizer","llm_provider","llm_model","temperature","top_p","max_tokens","seed","max_context_tokens","prompt_version","test_mode"]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame({n: {c: getattr(cfg, c) for c in cols} for n, cfg in CONFIGS.items()}).T

## 4. Ingesta e indexación
El sistema lee automáticamente todos los archivos de `corpus/` al iniciar.

In [ ]:
CORPUS = "corpus"
sistemas = {}
for nombre in ["R0", "R1", "R2"]:
    sistemas[nombre] = RAGSystem(CONFIGS[nombre], CORPUS)
    print(nombre, {k: v for k, v in sistemas[nombre].build_info.items() if k != "corpus"})
pd.DataFrame(sistemas["R0"].build_info["corpus"])

### Fragmentos y localizadores: R0 (fija 400/60) frente a R1 (estructural)

In [ ]:
def tabla_fragmentos(s):
    return pd.DataFrame([{"id": c.chunk_id, "tokens": c.n_tokens, "localizador": c.locator,
                          "inicio": c.text[:90].replace("\n", " ")} for c in s.chunks])
display(tabla_fragmentos(sistemas["R0"]))
display(tabla_fragmentos(sistemas["R1"]))

## 5. Demo de consulta
Muestra la respuesta, los fragmentos recuperados en orden, las citas (archivo + localizador) y el aviso de evidencia insuficiente.

In [ ]:
def preguntar(pregunta, config="R0"):
    s = sistemas.get(config) or RAGSystem(CONFIGS[config], CORPUS)
    print(format_result(s.ask(pregunta)))

preguntar("¿Qué debo hacer si el cliente VPN me muestra el error VPN-809?", "R0")

In [ ]:
preguntar("¿Cuál es el horario de atención de la Mesa de Ayuda y qué pasa con un incidente crítico un domingo?", "R2")

In [ ]:
preguntar("¿Cuánto cuesta la licencia del cliente VPN NovaConnect por usuario?", "R0")  # debe abstenerse

## 6. Preguntas de evaluación (fijadas antes de las corridas)

In [ ]:
import json
preg = json.load(open("evaluacion/preguntas.json", encoding="utf-8"))["preguntas"]
pd.DataFrame([{"id": q["id"], "tipo": q["tipo"], "pregunta": q["pregunta"], "respuesta esperada": q["respuesta_esperada"],
               "evidencia": " | ".join(f'{e["archivo"]} · {e["ubicacion"]}' for e in q["evidencia"]) or q.get("justificacion")}
              for q in preg])

In [ ]:
!python scripts/verificar_evidencia.py

### Hipótesis
Las hipótesis están en `HIPOTESIS.md` y se registraron **antes** de ejecutar las corridas finales. Resumen:
- **R1 vs R0:** evidencia menos cortada y con menos ruido → mejor respaldo de citas en directas; posible pérdida de cobertura en multidocumento (menos texto total con k=4).
- **R2 vs R0:** BM25 ayuda con códigos exactos (VPN-809, VPN-401) → mejor cobertura; riesgo de distractores en preguntas sin respuesta.
- **C0 vs R0:** C0 casi nunca acierta datos internos; riesgo de respuestas genéricas en lugar de abstenerse.

## 7. Parte A — 9 preguntas × 4 configuraciones = 36 corridas
Reanudable: si se interrumpe por límite de peticiones, vuelva a ejecutar la celda.

In [ ]:
from rag.evaluation import run_evaluation, export_valoracion, compute_metrics
SALIDA_A = "resultados/parte_A"
run_evaluation(["R0", "R1", "R2", "C0"], "evaluacion/preguntas.json", CORPUS, SALIDA_A)
export_valoracion(SALIDA_A)

In [ ]:
regs = pd.read_json(f"{SALIDA_A}/registros.jsonl", lines=True)
regs[["configuracion","id_pregunta","tipo","abstencion","contexto_recortado","tiempo_total_s","modelo_servido","respuesta"]]

## 8. Valoración manual y métricas
1. Abra `resultados/parte_A/valoracion.csv` (Excel o Google Sheets).
2. Para cada fila lea la respuesta, las citas y (si hace falta) el contexto en `registros.jsonl`.
3. Escriba **1/0** en `correcta_manual`, `respaldo_manual`, `abstencion_manual` y `sustantiva_manual` cuando la pre-valoración automática esté vacía o sea incorrecta, y deje un comentario.
4. Guarde el CSV (con el mismo nombre) y ejecute la celda siguiente. La columna `valores_solo_auto` de la tabla indica cuántas celdas faltan por revisar.

In [ ]:
tabla_A = pd.DataFrame(compute_metrics(SALIDA_A, ["R0", "R1", "R2", "C0"]))
tabla_A

In [ ]:
# Detalle de cobertura por pregunta (útil para el análisis R1 vs R0 y R2 vs R0)
cov = regs[regs.configuracion != "C0"].pivot_table(index="id_pregunta", columns="configuracion",
        values="cobertura_recuperados_auto", aggfunc="first")
cov

In [ ]:
# Posición (rank) del fragmento que contiene cada pasaje de referencia, por configuración
from rag.evaluation import norm
def rank_evidencia(row):
    q = next(q for q in preg if q["id"] == row.id_pregunta)
    out = []
    for ev in q["evidencia"]:
        rk = next((r["rank"] for r in row.recuperados if norm(ev["texto"]) in norm(r["texto"])), None)
        out.append(rk if rk else "✘")
    return out
regs_rag = regs[(regs.configuracion != "C0") & (regs.tipo_pregunta != "sin_respuesta")].copy()
regs_rag["rank_pasajes"] = regs_rag.apply(rank_evidencia, axis=1)
regs_rag.pivot_table(index="id_pregunta", columns="configuracion", values="rank_pasajes", aggfunc="first")

## 9. Parte B — otro modelo de embeddings y otro LLM (sobre R0)
B1 reconstruye el índice con `BAAI/bge-m3` y codifica las consultas con el mismo modelo; B2 cambia solo el LLM; B3 cambia ambos.

In [ ]:
SALIDA_B = "resultados/parte_B"
run_evaluation(["B1", "B2", "B3"], "evaluacion/preguntas.json", CORPUS, SALIDA_B)
export_valoracion(SALIDA_B)

In [ ]:
# Revise también valoracion.csv de la parte B; luego:
tabla_B = pd.DataFrame(compute_metrics(SALIDA_B, ["B1", "B2", "B3"]))
pd.concat([tabla_A[tabla_A.configuracion == "R0"], tabla_B])

## 10. Análisis (completar con los resultados)
- **R1 vs R0:** …
- **R2 vs R0:** …
- **C0 vs R0:** …
- **Experimento B:** …
- **Configuración recomendada:** …
- **Límites:** semilla no garantizada por el proveedor, rotación de modelos gratuitos, tamaño de la muestra (9 preguntas).

Copie la tabla final (`tabla_resultados.md`) y estas conclusiones al informe.